# 05 — Gradient Boosting Model (LightGBM)

**Synopsis Objective 4 — predictive model development (boosting half).** Notebook 04 covered the two linear/zero-training members of the model family: the seasonal-naive and moving-average baselines, and a Ridge regression. This notebook adds the gradient-boosted tree model, LightGBM, to the same comparison.

The goal here is the same as in notebook 04: fit the model on one illustrative train/validation split and sanity-check that it produces sensible, evaluable predictions on the Sales scale. This is **not** the formal model-selection step — with a single split, one fold's validation window could happen to favor or disadvantage any given model by chance. The rigorous comparison across `n_folds=5` walk-forward folds, with all models evaluated on identical splits, is notebook 06's job. Here we only need enough evidence to confirm the LightGBM pipeline is wired correctly end to end: it trains on the engineered feature matrix from notebook 03, respects the no-leakage lag/rolling construction, and produces metrics in the same `rmse / mae / wmape / mase` vocabulary as every other model in this project.

We reuse the exact same `FEATURE_COLS` contract (`NUMERIC_FEATURE_COLS + CATEGORICAL_FEATURE_COLS` from `src/features.py`) that the Ridge pipeline used, so any difference in illustrative performance here traces to the model class, not to the inputs it was given.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

## Imports

Alongside the usual `src` helpers, this notebook pulls in `lightgbm` directly (as `lgb`) for exactly one reason: the `early_stopping` callback used below lives on the `lightgbm` module itself, not on the `LGBMRegressor` estimator that `build_lightgbm_model` returns. Everything else — the model builder, the categorical-dtype caster, the single-split helper, and the shared metric bundle — comes from `src`, matching notebook 04's imports wherever the two notebooks share a step.

In [ ]:
import numpy as np
import lightgbm as lgb
from src.data_io import load_parquet
from src.features import NUMERIC_FEATURE_COLS, CATEGORICAL_FEATURE_COLS
from src.models import build_lightgbm_model, set_categorical_dtypes
from src.validation import walk_forward_splits
from src.metrics import evaluate_all

## Load the engineered feature matrix

`train_features.parquet` is the output of notebook 03's `build_feature_matrix`, already filtered to trainable rows (`Open == 1` and every lag/rolling column populated) and already carrying `log_sales`. We re-sort by `(Store, Date)` before splitting — the walk-forward and lag/rolling logic are all position-and-date sensitive, so this guards against the parquet round-trip having reshuffled row order.

`FEATURE_COLS` is the identical concatenation of `NUMERIC_FEATURE_COLS` and `CATEGORICAL_FEATURE_COLS` used by the Ridge pipeline in notebook 04. Keeping this definition identical across notebooks is what makes the eventual notebook 06 comparison a fair one: every model family sees the same inputs, and only the model class and its handling of those inputs differs.

In [ ]:
df = load_parquet("../data/processed/train_features.parquet").sort_values(["Store", "Date"]).reset_index(drop=True)
FEATURE_COLS = NUMERIC_FEATURE_COLS + CATEGORICAL_FEATURE_COLS

## One illustrative split, with native categorical columns

As in notebook 04, we take a single fold from `walk_forward_splits(df["Date"], n_folds=1, val_days=42)` — the most recent 42 days become the validation block, everything strictly before it is training. This is purely illustrative; it is one arbitrary 42-day window near the end of the series, and notebook 06 repeats this five times over expanding windows to get a result that isn't an artifact of which 42 days happened to be held out.

The one real difference from notebook 04 shows up in `set_categorical_dtypes`. The Ridge pipeline needed a `ColumnTransformer` with `OneHotEncoder` to turn `CATEGORICAL_FEATURE_COLS` into numeric columns Ridge could multiply against a coefficient vector. LightGBM doesn't need that: its sklearn API auto-detects pandas `category`-dtype columns at fit time and splits on them natively (as groupings of levels, found via its histogram-based algorithm), rather than requiring them to already be numeric. That matters a lot for this dataset specifically because `Store` is one of the five categorical columns and has **1,115 distinct levels** — one-hot encoding it would blow a single column up into 1,115 sparse columns, bloating memory and training time for no benefit to a tree model that can just ask "does this row's Store belong to {this subset of stores}?" at a split. `set_categorical_dtypes` does nothing more than `.astype("category")` on each of `CATEGORICAL_FEATURE_COLS`, applied separately to the train and validation slices defined by the fold's boolean masks.

In [ ]:
split = walk_forward_splits(df["Date"], n_folds=1, val_days=42)[0]
train_df = set_categorical_dtypes(df.loc[split["train_mask"]], CATEGORICAL_FEATURE_COLS)
val_df = set_categorical_dtypes(df.loc[split["val_mask"]], CATEGORICAL_FEATURE_COLS)

## Fit LightGBM with early stopping

`build_lightgbm_model()` returns an unfit `LGBMRegressor` configured with `objective="regression"`, `n_estimators=1000`, `learning_rate=0.05`, `num_leaves=63`, and 0.8 row/column subsampling — trained, like every model in this project, against `log_sales` rather than raw `Sales`, so its predictions have to be `expm1`-transformed back before they mean anything on the Sales scale.

`n_estimators=1000` is deliberately generous: it's an upper bound on boosting rounds, not a target. Left to run all 1000 rounds unconditionally, a model this flexible — 63-leaf trees, low learning rate encouraging many small corrective steps — would keep driving training loss down by fitting noise in `train_df` long after it had stopped learning anything generalizable, and validation error would start climbing back up. That's exactly what the `eval_set=[(val_df[FEATURE_COLS], val_df["log_sales"])]` argument combined with the `lgb.early_stopping(50, verbose=False)` callback is for: after every boosting round LightGBM checks validation loss, and if it hasn't improved in 50 consecutive rounds, training stops and the model keeps the best-performing iteration rather than the last one. This is the standard guard against overfitting a boosting model with a large `n_estimators` ceiling, and it's only honest to use here because `val_df` is genuinely out-of-time relative to `train_df` (the walk-forward split keeps it strictly after `split["train_end"]`) — so early stopping on it is a legitimate generalization signal, not training on the test set.

In [ ]:
model = build_lightgbm_model()
model.fit(
    train_df[FEATURE_COLS], train_df["log_sales"],
    eval_set=[(val_df[FEATURE_COLS], val_df["log_sales"])],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)

## Back-transform and score

`model.predict` returns predictions in log-space, so they go through `np.expm1` before touching any metric — the same discipline notebook 04 followed for the baselines and Ridge. `evaluate_all` then reports the shared `rmse / mae / wmape / mase` bundle against actual `Sales`.

`train_groups=train_df["Store"]` is passed for the same reason it was passed for the Ridge model: this is one global model fit across all 1,115 stores at once, not a separate model per store, so the naive-error scale inside `mase` must be computed per store and then pooled — differencing `Sales` at a store boundary (store 2's last day vs. store 3's first day) would produce a meaningless jump and corrupt the scale.

In [ ]:
pred = np.expm1(model.predict(val_df[FEATURE_COLS]))
metrics = evaluate_all(val_df["Sales"], pred, train_df["Sales"], seasonal_period=7, train_groups=train_df["Store"])
metrics

## Feature importance: a sanity check, not the final word

As a quick check that the model is actually learning from the right signal and not latching onto something spurious, we look at `model.feature_importances_` against `FEATURE_COLS`. One caveat worth stating plainly: `LGBMRegressor`'s default `importance_type` is `"split"` — the count of times a feature was used anywhere as a split point across all trees — not `"gain"`, the total reduction in loss it's responsible for. A feature can split often without contributing much to accuracy, so this ranking is a sanity check on *which columns the model is looking at*, not a final statement of *how much each one matters*. That more rigorous question is exactly what notebook 07's SHAP-based interpretation is for. Here, we just want to confirm the top of the list looks like what domain intuition would predict for retail demand — recent lag/rolling sales history, `Promo`, and calendar effects near the top — before carrying this model into the formal evaluation.

In [ ]:
import pandas as pd
importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
ax = importances.head(20).plot(kind="barh", figsize=(8, 8))
ax.invert_yaxis()
ax.set_title("LightGBM Feature Importance (illustrative single split)")

## Takeaway

LightGBM trains cleanly on the shared `FEATURE_COLS` contract, native categorical columns included, and early stopping kept it from riding all 1,000 boosting rounds into overfitting against this one validation window. The metrics above are a single data point, not a verdict — notebook 06 runs this same model (and the baselines and Ridge from notebook 04) across all five walk-forward folds and aggregates the results, which is the only basis on which this project draws conclusions about which model actually generalizes best for store-level demand forecasting.